# NB02 · ¿Con qué modelo? — modelos, prefijos, normalización, métrica

**D09 ya está decidida** (`config/config.yaml`): compiten `gemini-embedding-2`, `jinaai/jina-embeddings-v3` e `ibm-granite/granite-embedding-311m-multilingual-r2`, con la **plantilla congelada en A0** (la columna `text` del dataset, tal cual).

Este notebook empieza por el **paso 0**: antes de codificar nada, medir **cuánto texto ve realmente cada candidato**. Es la evidencia que decide **D07** (¿hace falta chunking?) y la que verifica que la ventana de contexto de los tres modelos cubre el catálogo.

---

### ⚠️ Por qué las longitudes de NB01 no sirven aquí

NB01 midió `text` en **palabras**, con la expresión regular de `aurum.datos.tokenize` (p50 = 150). El límite de contexto de un modelo se mide en **piezas de su vocabulario de subpalabras**, que es otra unidad: una palabra larga o un código como `160x200` se lleva varias piezas. La única cuenta válida es la del tokenizador de cada modelo, y por eso se descarga aquí.

| Marca | Corpus | Fichero |
|---|---|---|
| 🔬 **MUESTRA** | 1.500 registros | `catalogo_muestra.csv` |
| 📚 **COMPLETO** | 15.000 registros | `catalogo_productos.csv` |

In [12]:
import os
import sys
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")
sys.path.insert(0, str(Path("..") / "src"))

import pandas as pd
from dotenv import load_dotenv
from transformers import logging as hf_logging

from aurum.embeddings import load_hub_tokenizer, token_length_report, token_lengths

hf_logging.set_verbosity_error()
# Carga HF_TOKEN y GEMINI_API_KEY en el entorno del proceso. El fichero .env no se
# imprime ni se versiona: solo se leen los valores desde os.environ.
load_dotenv(Path("..") / ".env")

DATA = Path("..") / "data"
muestra = pd.read_csv(DATA / "catalogo_muestra.csv")
completo = pd.read_csv(DATA / "catalogo_productos.csv")

print(f"🔬 MUESTRA : {len(muestra):>6} registros")
print(f"📚 COMPLETO: {len(completo):>6} registros")
print(f"HF_TOKEN cargado: {bool(os.environ.get('HF_TOKEN'))}")


🔬 MUESTRA :   1500 registros
📚 COMPLETO:  15000 registros
HF_TOKEN cargado: True


## A.1 · Los tres candidatos de D09

Datos de cada uno, contrastados contra su *model card* y su `config.json` (no contra la memoria de nadie). `gemini-embedding-2` es API, así que no tiene tokenizador descargable: su ventana se toma de la documentación oficial.

| Modelo | Dim nativa | Ventana | Contrato de entrada | Acceso |
|---|---:|---:|---|---|
| `gemini-embedding-2` | 3.072 (MRL 128–3.072) | 8.192 | instrucción **en el prompt** — no admite `task_type` | API de Google (`GEMINI_API_KEY`) |
| `jinaai/jina-embeddings-v3` | 1.024 (MRL 32–1.024) | 8.192 | **adaptadores LoRA por tarea**: `retrieval.query` · `retrieval.passage` · `text-matching` | HF abierto, licencia **`cc-by-nc-4.0`** ⚠️ |
| `ibm-granite/granite-embedding-311m-multilingual-r2` | 768 | 32.768 | por confirmar en su model card | HF abierto (Apache-2.0) |

In [13]:
CANDIDATOS_HF = {  # espejo de config.yaml -> nb02_modelo.d09_modelos
    "jina-v3": ("jinaai/jina-embeddings-v3", 8192),
    "granite-311m-r2": ("ibm-granite/granite-embedding-311m-multilingual-r2", 32768),
}
VENTANA_GEMINI = 8192  # gemini-embedding-2, según la documentación de la API

# `load_hub_tokenizer` baja solo el tokenizer.json: jina-v3 lleva código propio en
# el repo y AutoTokenizer exigiría trust_remote_code, que importa torch. Para
# contar tokens basta el vocabulario.
tokenizadores = {}
for alias, (repo, ventana) in CANDIDATOS_HF.items():
    try:
        tokenizadores[alias] = (
            load_hub_tokenizer(repo, token=os.environ.get("HF_TOKEN")),
            ventana,
        )
        print(f"✅ {alias}: tokenizador descargado")
    except Exception as error:  # sin red, repo gated o token sin permiso
        print(f"⛔ {alias}: {type(error).__name__} — {str(error).splitlines()[0]}")


✅ jina-v3: tokenizador descargado
✅ granite-311m-r2: tokenizador descargado


## A.2 · 🔬 Longitud en tokens sobre la muestra

`pct_supera_ventana` es el número que decide **D07**. `chars_por_token` mide cuánto se aleja la cuenta real de una estimación en caracteres.

In [3]:
token_length_report(muestra["text"], tokenizadores)


,modelo,ventana,n_docs,tokens_media,tokens_p50,tokens_p90,tokens_p95,tokens_max,n_supera_ventana,pct_supera_ventana,margen_p95,chars_por_token
0,jina-v3,8192,1500,362.1,293,782,835,1308,0,0.0,7357,3.61
1,granite-311m-r2,32768,1500,352.7,278,754,811,1875,0,0.0,31957,3.71


## A.3 · 📚 La misma medición sobre el catálogo completo

⏱️ Tarda ~30 s: tokeniza 15.000 registros.

In [4]:
informe_completo = token_length_report(completo["text"], tokenizadores)
informe_completo


,modelo,ventana,n_docs,tokens_media,tokens_p50,tokens_p90,tokens_p95,tokens_max,n_supera_ventana,pct_supera_ventana,margen_p95,chars_por_token
0,jina-v3,8192,15000,334.1,250,769,820,1308,0,0.0,7372,3.63
1,granite-311m-r2,32768,15000,322.2,242,734,789,1972,0,0.0,31979,3.76


## A.4 · 📚 Cuántos registros del catálogo se truncarían con cada tamaño de ventana

La pregunta de fondo de D07 no es *"¿se trunca?"* sino *"¿a partir de qué ventana deja de truncarse?"*. Esta tabla la responde de una vez para cualquier modelo, presente o futuro.

In [4]:
alias_referencia = next(iter(tokenizadores))
longitudes = token_lengths(completo["text"], tokenizadores[alias_referencia][0])

pd.DataFrame([
    {
        "ventana": ventana,
        "registros_que_la_superan": int((longitudes > ventana).sum()),
        "pct": round(100 * float((longitudes > ventana).mean()), 2),
    }
    for ventana in (128, 512, 1024, 2048, 8192)
]).assign(tokenizador=alias_referencia)


,ventana,registros_que_la_superan,pct,tokenizador
0,128,10006,66.71,jina-v3
1,512,4167,27.78,jina-v3
2,1024,25,0.17,jina-v3
3,2048,0,0.00,jina-v3
4,8192,0,0.00,jina-v3


## A.5 · 📚 `gemini-embedding-2`: medición contra la API

Los dos modelos locales se miden con su tokenizador descargado. Gemini no publica el suyo, pero la API expone `count_tokens` **para el propio modelo de embeddings** — así que no hay que estimar nada ni usar un modelo generativo como sustituto.

⚠️ **Es una petición de red por registro del catálogo**, así que se mide un subconjunto en vez de las 15.000: las **50 registros más largos** en caracteres —que son las únicas que podrían acercarse a la ventana— más **100 al azar** para el ratio `chars_por_token`. Con un máximo local de 1.972 tokens contra una ventana de 8.192, el margen es de 4×: no hace falta más precisión para responder a D07.

Si no hay `GEMINI_API_KEY`, la celda se salta sin romper el notebook — el corrector puede ejecutar el resto sin clave.

In [12]:
from aurum.embeddings import CountingTokenizer, gemini_token_counter

MODELO_GEMINI = "gemini-embedding-2"
N_MAS_LARGAS, N_AZAR = 50, 100

longitud_chars = completo["text"].fillna("").str.len()
mas_largas = completo.loc[longitud_chars.nlargest(N_MAS_LARGAS).index, "text"]
al_azar = completo["text"].sample(N_AZAR, random_state=42)
subconjunto = pd.concat([mas_largas, al_azar]).drop_duplicates()

clave = os.environ.get("GEMINI_API_KEY")
if not clave:
    print("⏭️  Sin GEMINI_API_KEY: se omite la medición de gemini-embedding-2")
else:
    gemini = CountingTokenizer(gemini_token_counter(MODELO_GEMINI, api_key=clave))
    informe_gemini = token_length_report(subconjunto, {MODELO_GEMINI: (gemini, VENTANA_GEMINI)})
    display(informe_gemini)


,modelo,ventana,n_docs,tokens_media,tokens_p50,tokens_p90,tokens_p95,tokens_max,n_supera_ventana,pct_supera_ventana,margen_p95,chars_por_token
0,gemini-embedding-2,8192,150,464.6,530,772,798,1887,0,0.0,7394,3.94


### A.5b · 📚 El mismo subconjunto con los tres tokenizadores

Comparar los tres sobre **los mismos registros** es lo que exige la Regla 2: si cada modelo se midiera sobre un corpus distinto, las columnas no serían comparables entre sí.

In [7]:
if clave:
    todos = {MODELO_GEMINI: (gemini, VENTANA_GEMINI), **tokenizadores}
    display(token_length_report(subconjunto, todos))


,modelo,ventana,n_docs,tokens_media,tokens_p50,tokens_p90,tokens_p95,tokens_max,n_supera_ventana,pct_supera_ventana,margen_p95,chars_por_token
0,gemini-embedding-2,8192,150,464.6,530,772,798,1887,0,0.0,7394,3.94
1,jina-v3,8192,150,499.7,564,849,862,1308,0,0.0,7330,3.66
2,granite-311m-r2,32768,150,483.5,546,811,835,1875,0,0.0,31933,3.78


## A.6 · Cómo se lee esto para D07

El chunking (familia C del plan) solo tiene sentido si el modelo elegido **no puede leer el registro del catálogo entero**. Con `pct_supera_ventana = 0` en los tres candidatos, no hay información que se pierda por truncado y la familia C queda descartada **por medición**, no por falta de tiempo.

Consecuencia directa en la base vectorial: el punto sigue siendo `record_id` (relación 1:1 producto↔vector), el esquema de NB04 se mantiene simple y la idempotencia no necesita borrar chunks huérfanos.

---

# B · Codificación de los tres candidatos

Hasta aquí se ha medido **cuánto texto ve** cada modelo. Ahora se codifica de verdad, sobre `catalogo_muestra.csv` (condición 3 del plan: la muestra existe justo para esto; el catálogo completo solo se ingiere en la ejecución final).

### Qué se codifica y qué no

| Eje de D10 | ¿Obliga a recodificar? | Cómo se barre |
|---|---|---|
| **Modelo** | Sí | 3 codificaciones |
| **Contrato de entrada** (con/sin prefijos) | Sí | ×2 **solo** en los modelos que tienen contrato |
| **Dimensión (MRL)** | No | Truncar + renormalizar los mismos vectores |
| **Normalización L2** | No | Post-proceso |
| **Métrica** (`cosine`·`dot`·`l2`) | No | Cambia el buscador, no los vectores |

Los tres últimos ejes son **gratis**, y por eso el barrido de dimensión de la sección C cubre 15 configuraciones sin pagar 15 codificaciones.

### ⏱️ Lo que esto cuesta en esta máquina

4 núcleos, sin GPU. `jina-embeddings-v3` son 572M de parámetros: cargarlo en `float32` ya ocupa ~2,3 GB de los 7,9 GB disponibles, así que **los modelos se cargan y se liberan de uno en uno**. Cuenta con decenas de minutos la primera vez.

La segunda vez es instantánea: `encode_corpus` guarda los vectores en `artifacts/embeddings/` junto a un `.json` con el `model_id`, la dimensión, el dtype y el **SHA-256 del corpus**. Si el texto cambia (por ejemplo al pasar de la plantilla A0 a otra en NB03), la huella cambia y la caché se invalida sola — que es lo que impide comparar en silencio vectores de dos textos distintos.

In [14]:
import gc
import time

import torch

from aurum.busqueda import DenseRetriever, rank_queries_dense
from aurum.embeddings import (
    GeminiEncoder,
    SentenceTransformerEncoder,
    encode_corpus,
    safe_l2_normalize,
    truncate_dim,
    vector_health,
)
from aurum.evaluacion import apply_tolerance_rule, evaluate_rankings, qrels_from_judgements
from aurum.graficas import (
    plot_contract_delta,
    plot_dimension_curve,
    plot_metric_comparison,
)

torch.set_num_threads(4)  # los 4 núcleos físicos de la máquina

CORPUS_ID = "catalogo_muestra"   # condición 3 del plan
PLANTILLA = "A0"                 # congelada: la columna `text` tal cual
CAMPO = "text"
TOP_K = 10
BATCH_LOCAL = 8                  # 8 GB de RAM con un modelo de 572M cargado
CACHE = Path("..") / "artifacts" / "embeddings"
TOLERANCIA_D09B = 0.02           # tau de config.yaml -> d09b_criterio_desempate

consultas = pd.read_csv(DATA / "consultas_desarrollo.csv")
relevancias = pd.read_csv(DATA / "relevancias_desarrollo.csv")
qrels = qrels_from_judgements(relevancias)

corpus_textos = muestra[CAMPO].tolist()
corpus_ids = muestra["product_id"].tolist()
query_ids = [str(q) for q in consultas["query_id"]]
query_textos = consultas["query_text"].tolist()

print(f"corpus   : {len(corpus_textos)} documentos (plantilla {PLANTILLA})")
print(f"consultas: {len(query_textos)} de desarrollo, {len(qrels)} juzgadas")

corpus   : 1500 documentos (plantilla A0)
consultas: 8 de desarrollo, 8 juzgadas


## B.1 · Registro de modelos

Cada ficha reproduce lo verificado en A.1 más lo que el modelo necesita para codificar. La columna que más decisiones arrastra es **el contrato de entrada**:

| Modelo | Mecanismo del contrato | ¿Entra en el eje con/sin de D10? |
|---|---|---|
| `jina-v3` | **Adaptadores LoRA** (`retrieval.passage` / `retrieval.query`) — pesos distintos, no texto | ✅ Sí, y cuesta ×2 codificaciones |
| `granite-311m-r2` | **Ninguno** — confirmado en la model card de IBM | ❌ No: no hay contrato que retirar |
| `gemini-2` | **Instrucción dentro del prompt** | ✅ Sí (API, barato) |

> 🔎 **Por qué granite no entra en ese eje, y por qué es provisional.** Su `config_sentence_transformers.json` declara literalmente `"prompts": {"query": "", "document": ""}`: los dos prompts son la cadena vacía. Codificar "con contrato" y "sin contrato" daría **exactamente los mismos vectores**, así que la Δ sería 0 por construcción y gastar dos codificaciones en demostrarlo no es evidencia, es tiempo de CPU. Inventarle un prefijo sería peor: estaríamos midiendo un modelo que nadie entrenó.
>
> Ese argumento se apoyaba en un fichero del repositorio, y §3.1 avisa de que *"no basta con citar la documentación del modelo"*: el fichero prueba que **la librería** no antepone nada, no que **IBM** entrenara el modelo sin instrucción. Quedó registrado como **P02** y **está cerrado**. Revisada la model card completa —todos los backends documentados (`sentence-transformers`, Transformers, ONNX, OpenVINO, vLLM, GGUF) más las secciones *Usage* y *When to Use This Model*—, **no hay ninguna instrucción ni prefijo en ningún sitio**: en el ejemplo de retrieval de IBM, consultas y documentos se pasan por igual a `model.encode()`. El contrato real es **texto plano simétrico**, confirmado por la fuente primaria y no por inferencia de un JSON. `granite` no compitió en desventaja.

> ⚠️ `jina-v3` exige `trust_remote_code=True`: se ejecuta código del repositorio de Jina. Es una consecuencia que hereda el corrector y queda anotada en el README. Su licencia **`cc-by-nc-4.0`** es además una de las *"restricciones del caso"* que §3.1 obliga a pesar en la elección: prohíbe el uso comercial, que es exactamente el escenario de un marketplace.

In [15]:
REGISTRO = {
    "jina-v3": {
        "repo": "jinaai/jina-embeddings-v3",
        "ventana": 8192,
        "dim_nativa": 1024,
        "tasks": {"document": "retrieval.passage", "query": "retrieval.query"},
        "trust_remote_code": True,
        "dims": [1024, 768, 512, 256, 128],
        "licencia": "cc-by-nc-4.0",
    },
    "granite-311m-r2": {
        "repo": "ibm-granite/granite-embedding-311m-multilingual-r2",
        "ventana": 32768,
        "dim_nativa": 768,
        "tasks": None,  # prompts declarados como cadena vacía: no hay contrato
        "trust_remote_code": False,
        "dims": [768, 512, 256, 128],
        "licencia": "apache-2.0",
    },
    "gemini-2": {
        "repo": "gemini-embedding-2",
        "ventana": VENTANA_GEMINI,
        "dim_nativa": 3072,
        "api": True,
        "dims": [3072, 1536, 768, 512, 256, 128],
        "licencia": "servicio de terceros",
    },
}


def fabricar(alias):
    """Construye el encoder del alias. Se llama justo antes de codificar y el
    objeto se libera después: dos modelos locales a la vez no caben en 8 GB."""
    ficha = REGISTRO[alias]
    if ficha.get("api"):
        return GeminiEncoder(
            api_key=os.environ.get("GEMINI_API_KEY"),
            model_id=ficha["repo"],
            native_dim=ficha["dim_nativa"],
            window=ficha["ventana"],
        )
    return SentenceTransformerEncoder(
        ficha["repo"],
        window=ficha["ventana"],
        native_dim=ficha["dim_nativa"],
        tasks=ficha["tasks"],
        trust_remote_code=ficha["trust_remote_code"],
        device="cpu",
        token=os.environ.get("HF_TOKEN"),
    )


pd.DataFrame([
    {
        "alias": alias,
        "repo": f["repo"],
        "dim_nativa": f["dim_nativa"],
        "ventana": f["ventana"],
        "tiene_contrato": bool(f.get("api") or f.get("tasks")),
        "licencia": f["licencia"],
    }
    for alias, f in REGISTRO.items()
])


,alias,repo,dim_nativa,ventana,tiene_contrato,licencia
0,jina-v3,jinaai/jina-embeddings-v3,1024,8192,True,cc-by-nc-4.0
1,granite-311m-r2,ibm-granite/granite-embedding-311m-multilingua...,768,32768,False,apache-2.0
2,gemini-2,gemini-embedding-2,3072,8192,True,servicio de terceros


## B.2 · Codificar — ⏱️ **una celda por modelo, ejecutables por separado**

Los tres modelos **no** se codifican en un bucle: cada uno tiene su celda y se puede lanzar por su cuenta, en el orden que quieras y en sesiones distintas. Hay tres razones y ninguna es estética:

1. **RAM.** `jina-v3` en `float32` ocupa ~2,3 GB de los 7,9 GB de la máquina. Cada celda construye el modelo, codifica y lo libera con `gc.collect()` antes de devolver el control. Dos modelos vivos a la vez no caben.
2. **Tiempo.** Son decenas de minutos por modelo. Un bucle único obliga a esperar a los tres para ver el primer número.
3. **Aislamiento de fallos.** Si `jina-v3` revienta por su `trust_remote_code` o Gemini se queda sin cuota, los demás ya están medidos. El enunciado pide *"al menos dos configuraciones relevantes"*: con dos de tres sigues teniendo comparación.

### 🔑 Cada celda codifica las DOS variantes del modelo

`nativo` (aplicando el contrato de entrada que el modelo declara) y `sin_contrato` (omitiéndolo). Las dos llamadas están juntas a propósito, y no es una comodidad: es lo que hace que **el notebook dé el mismo resultado se ejecute como se ejecute**.

El barrido de la sección **C** evalúa todo lo que encuentre en `VECTORES`. Si `sin_contrato` se codificara más abajo —en la sección D, que es donde se analiza—, quien ejecutara el notebook de principio a fin llegaría a C con solo la mitad de las configuraciones, y **C.1, C.2, F y G decidirían sobre media tabla sin mostrar ningún aviso**. Codificando aquí las dos ramas, el orden de ejecución deja de importar.

La sección D, por tanto, **no codifica nada**: solo mide la diferencia entre las dos ramas que ya existen.

> `granite-311m-r2` no tiene contrato que retirar, así que su segunda llamada no codifica nada y lo dice por pantalla. Es documentación ejecutable: en el informe se ve que se saltó a propósito y no por olvido (**P02**, cerrado con la model card).

### Cómo se combinan después

Cada celda deposita sus vectores en `VECTORES`, indexado por `(modelo, contrato)`. **Todo lo que viene detrás (barrido C, contrato D, métrica E, comparación F, regla G) lee ese diccionario y trabaja con lo que encuentre**, sea uno, dos o tres modelos. No hace falta que estén los tres para obtener métricas: las tablas saldrán con las filas que haya.

> 🔁 **Tras reiniciar el kernel** vuelve a ejecutar las tres celdas: como los vectores están en `artifacts/embeddings/`, la segunda vez son segundos, no minutos. Re-ejecutar una celda tampoco duplica nada — `COSTES` está indexado por `(modelo, contrato, tipo)`, así que sobrescribe en vez de acumular.

In [16]:
VECTORES = {}   # (alias, contrato) -> {"document": ndarray, "query": ndarray}
COSTES = {}     # (alias, contrato, tipo) -> fila de coste. Dict, no lista: así
                # re-ejecutar la celda de un modelo sobrescribe en vez de duplicar.
ERRORES = {}


def codificar(alias, contrato="nativo"):
    """Codifica documentos y consultas de un modelo, y libera la memoria."""
    ficha = REGISTRO[alias]
    encoder = fabricar(alias)
    lotes = 32 if ficha.get("api") else BATCH_LOCAL
    salida = {}
    try:
        for kind, textos, corpus_id in (
            ("document", corpus_textos, CORPUS_ID),
            ("query", query_textos, "consultas_desarrollo"),
        ):
            resultado = encode_corpus(
                encoder, textos, corpus_id=corpus_id, kind=kind,
                contract=contrato, batch_size=lotes, cache_dir=CACHE,
            )
            salida[kind] = resultado.vectors
            COSTES[(alias, contrato, kind)] = {"alias": alias, **resultado.stats.as_row()}
    finally:
        # El `finally` importa: si la codificación de consultas falla, el modelo
        # se libera igual y el kernel no se queda con 2,3 GB retenidos.
        del encoder
        gc.collect()
    return salida


def ejecutar(alias, contrato="nativo"):
    """Codifica un modelo dejando el resultado en VECTORES, sin propagar el fallo.

    Cada celda de modelo llama aquí dos veces, una por rama de contrato. Un error
    se registra y se muestra, pero no detiene el notebook: los modelos que sí
    funcionaron siguen siendo medibles."""
    ficha = REGISTRO[alias]
    if contrato == "sin_contrato" and not (ficha.get("api") or ficha.get("tasks")):
        print(f"⏭️  {alias}: no tiene contrato de entrada, el eje no aplica")
        return
    inicio = time.perf_counter()
    try:
        VECTORES[(alias, contrato)] = codificar(alias, contrato)
        ERRORES.pop(f"{alias}[{contrato}]", None)
        print(f"✅ {alias} [{contrato}] listo en {time.perf_counter() - inicio:.1f}s")
    except Exception as error:
        ERRORES[f"{alias}[{contrato}]"] = f"{type(error).__name__}: {error}"
        print(f"⛔ {alias} [{contrato}]: {ERRORES[f'{alias}[{contrato}]'][:250]}")


def estado():
    """Qué hay codificado ahora mismo. Es lo que podrán medir las secciones C-G."""
    filas = [
        {
            "modelo": alias,
            "contrato": contrato,
            "docs": VECTORES[(alias, contrato)]["document"].shape,
            "consultas": VECTORES[(alias, contrato)]["query"].shape,
        }
        for (alias, contrato) in sorted(VECTORES)
    ]
    return pd.DataFrame(filas) if filas else "Todavía no hay ningún modelo codificado."


print(
    "Listo. Ejecuta las tres celdas siguientes en el orden que prefieras.\n"
    "Cada una codifica su modelo en las DOS ramas de contrato: la sección D las\n"
    "compara, pero no las genera, así que ninguna sección posterior depende del\n"
    "orden en que ejecutes esto."
)

Listo. Ejecuta las tres celdas siguientes en el orden que prefieras.
Cada una codifica su modelo en las DOS ramas de contrato: la sección D las
compara, pero no las genera, así que ninguna sección posterior depende del
orden en que ejecutes esto.


### B.2a · `jina-v3` — ⏱️ el más caro (572M de parámetros)

**Dos codificaciones completas de los 1.500 documentos.** En jina el contrato no es un prefijo de texto sino un **adaptador LoRA**: `retrieval.passage` para documentos y `retrieval.query` para consultas. Retirarlo significa usar otros pesos, así que la variante `sin_contrato` no se puede derivar de la primera — hay que codificar de nuevo. Es el eje más caro de todo D10, y por eso se paga aquí una sola vez.

⚠️ Ejecuta código del repositorio de Jina (`trust_remote_code=True`) y su licencia es **`cc-by-nc-4.0`**: no bloquea el experimento académico, pero sí la recomendación final para un marketplace real. Queda anotado para el informe.

In [17]:
ejecutar("jina-v3")                            # con contrato: adaptador LoRA por tarea
ejecutar("jina-v3", contrato="sin_contrato")   # sin él: mismos textos, otros pesos

Loading weights: 100%|██████████| 492/492 [00:11<00:00, 41.60it/s] 


✅ jina-v3 [nativo] listo en 55.6s


Loading weights: 100%|██████████| 492/492 [00:07<00:00, 66.14it/s] 


✅ jina-v3 [sin_contrato] listo en 29.1s


### B.2b · `granite-311m-r2` — Apache-2.0, sin código remoto

**Una sola codificación.** La segunda llamada está puesta pero no codifica: granite declara sus dos prompts como cadena vacía, así que `nativo` y `sin_contrato` producirían vectores idénticos. La celda imprime el motivo del salto en lugar de gastar otra pasada en demostrar una Δ que es 0 por construcción.

Esa exclusión se registró como **P02** y está **cerrada**: la model card completa de IBM no documenta ninguna instrucción ni prefijo en ningún backend, así que el contrato real es texto plano simétrico. La conclusión se sostiene ahora en la fuente primaria, no en un JSON de configuración.

In [18]:
ejecutar("granite-311m-r2")
# No codifica nada: granite no declara contrato que retirar. La llamada se deja
# puesta para que el salto aparezca en la salida del notebook — así el informe
# muestra que se omitió a propósito, no por olvido (P02, cerrado).
ejecutar("granite-311m-r2", contrato="sin_contrato")

Loading weights: 100%|██████████| 134/134 [00:00<00:00, 1401.18it/s]


✅ granite-311m-r2 [nativo] listo en 12.7s
⏭️  granite-311m-r2: no tiene contrato de entrada, el eje no aplica


### B.2c · `gemini-2` — por API

**Dos codificaciones, pero baratas.** En Gemini el contrato sí es texto: una instrucción de tarea antepuesta al contenido. Retirarla no cambia los pesos, solo lo que se envía, y como el trabajo lo hace la API el eje cuesta llamadas, no horas de CPU.

Necesita `GEMINI_API_KEY` en `.env`. Si no está, las dos llamadas fallan de forma controlada y el notebook continúa con los modelos locales.

In [19]:
ejecutar("gemini-2")                            # con la instrucción de tarea en el prompt
ejecutar("gemini-2", contrato="sin_contrato")   # con el texto desnudo

✅ gemini-2 [nativo] listo en 7.9s
✅ gemini-2 [sin_contrato] listo en 2.3s


### B.2d · Qué hay codificado y cuánto ha costado

El coste va **junto** a la calidad, no en una nota al pie: una ventaja de nDCG que se paga con 3× de tiempo de indexación es una decisión distinta a una ventaja gratis. Es el criterio que D09b declaró de antemano.

In [9]:
display(estado())
if ERRORES:
    display(pd.DataFrame([{"modelo": k, "error": v} for k, v in ERRORES.items()]))
pd.DataFrame(COSTES.values())


,modelo,contrato,docs,consultas
0,gemini-2,nativo,"(1500, 3072)","(8, 3072)"
1,gemini-2,sin_contrato,"(1500, 3072)","(8, 3072)"
2,granite-311m-r2,nativo,"(1500, 768)","(8, 768)"
3,jina-v3,nativo,"(1500, 1024)","(8, 1024)"
4,jina-v3,sin_contrato,"(1500, 1024)","(8, 1024)"


,alias,modelo,tipo,contrato,n_textos,dim,segundos,textos_por_segundo,desde_cache
0,jina-v3,jinaai/jina-embeddings-v3,document,nativo,1500,1024,4954.58,0.30,True
1,jina-v3,jinaai/jina-embeddings-v3,query,nativo,8,1024,4.06,1.97,True
2,jina-v3,jinaai/jina-embeddings-v3,document,sin_contrato,1500,1024,5751.93,0.26,True
3,jina-v3,jinaai/jina-embeddings-v3,query,sin_contrato,8,1024,1.61,4.97,True
4,granite-311m-r2,ibm-granite/granite-embedding-311m-multilingua...,document,nativo,1500,768,17489.67,0.09,True
5,granite-311m-r2,ibm-granite/granite-embedding-311m-multilingua...,query,nativo,8,768,1.83,4.37,True
6,gemini-2,gemini-embedding-2,document,nativo,1500,3072,45.41,33.03,True
7,gemini-2,gemini-embedding-2,query,nativo,8,3072,0.44,18.18,True
8,gemini-2,gemini-embedding-2,document,sin_contrato,1500,3072,49.59,30.25,True
9,gemini-2,gemini-embedding-2,query,sin_contrato,8,3072,0.43,18.60,True


## B.3 · Salud de los vectores — antes de creerse ninguna métrica

Un `NaN` o una matriz con filas repetidas producen métricas perfectamente presentables y perfectamente falsas. Estas comprobaciones son las que el plan exige en las *Métricas de verificación de NB02*: finitud, normas y duplicados.

**Atención a la columna `normalizado`, que resultó no ser un dato anecdótico.** `SentenceTransformerEncoder` pide `normalize_embeddings=False` a los dos modelos locales, así que cabría esperar que ninguno llegara unitario. No es lo que ocurre: `jina-v3` y `gemini-2` salen con norma exactamente 1 y solo `granite-311m-r2` entrega la salida cruda.

El motivo es que ese flag **añade** normalización cuando vale `True`; no **retira** un módulo `Normalize` que forme parte del modelo. El pipeline de `jina-v3` lleva el suyo, y Gemini devuelve vectores unitarios por API.

Eso convierte a `granite` en el único candidato con el que la sección E puede medir algo. Conviene tenerlo presente antes de leer aquella tabla.

In [10]:
pd.DataFrame([
    {"alias": alias, "contrato": contrato, "tipo": kind, **vector_health(matriz)}
    for (alias, contrato), por_tipo in VECTORES.items()
    for kind, matriz in por_tipo.items()
])


,alias,contrato,tipo,n_vectores,dim,dtype,finito,norma_min,norma_max,normalizado,n_filas_duplicadas,bytes_por_vector
0,jina-v3,nativo,document,1500,1024,float32,True,1.000000,1.000000,True,0,4096
1,jina-v3,nativo,query,8,1024,float32,True,1.000000,1.000000,True,0,4096
2,jina-v3,sin_contrato,document,1500,1024,float32,True,1.000000,1.000000,True,0,4096
3,jina-v3,sin_contrato,query,8,1024,float32,True,1.000000,1.000000,True,0,4096
4,granite-311m-r2,nativo,document,1500,768,float32,True,0.997481,1.002835,False,0,3072
5,granite-311m-r2,nativo,query,8,768,float32,True,0.998226,1.002176,False,0,3072
6,gemini-2,nativo,document,1500,3072,float32,True,1.000000,1.000000,True,0,12288
7,gemini-2,nativo,query,8,3072,float32,True,1.000000,1.000000,True,0,12288
8,gemini-2,sin_contrato,document,1500,3072,float32,True,1.000000,1.000000,True,0,12288
9,gemini-2,sin_contrato,query,8,3072,float32,True,1.000000,1.000000,True,0,12288


---

# C · Barrido de dimensión (MRL) — el eje gratis

Los tres candidatos están entrenados con **Matryoshka Representation Learning**: las primeras componentes concentran la mayor parte de la información, así que quedarse con un prefijo del vector es una reducción de dimensión válida y sin recodificar.

Truncar **obliga a renormalizar**: el prefijo de un vector unitario tiene norma < 1, y sin renormalizar el coseno deja de ser un coseno. `truncate_dim` lo hace siempre.

Lo que se busca en la tabla no es solo el máximo, sino **dónde se cae la curva**: si 256 dimensiones pierden menos de 0,02 de nDCG@10 frente a 1.024, el ahorro es de 4× en memoria del motor y en ancho de banda por consulta. Ese es exactamente el compromiso que D09b declaró de antemano.

In [20]:
def evaluar(vectores, dim, *, metric="cosine", normalizar=True):
    """Evalúa una configuración concreta sobre las 8 consultas de desarrollo.

    Devuelve el informe y los rankings: guardar los IDs y no solo la métrica es
    lo que permite atribuir errores en NB09 (Regla 3 de experimentación)."""
    docs = truncate_dim(vectores["document"], dim, renormalize=normalizar)
    queries = truncate_dim(vectores["query"], dim, renormalize=normalizar)
    retriever = DenseRetriever(docs, corpus_ids, metric=metric)
    rankings = rank_queries_dense(retriever, query_ids, queries, k=TOP_K)
    return evaluate_rankings(rankings, qrels, k=TOP_K), rankings


costes = pd.DataFrame(COSTES.values())
segundos_por_modelo = (
    costes.query("tipo == 'document'").set_index(["alias", "contrato"])["segundos"]
    if len(costes) else pd.Series(dtype=float)
)

if not VECTORES:
    raise RuntimeError(
        "No hay ningún modelo codificado: ejecuta B.2a, B.2b o B.2c antes de esta celda."
    )

# El barrido recorre TODO lo codificado: cada modelo en sus dos ramas de
# contrato. Filtrar aquí por `nativo` haría que la regla D09b de la sección G
# eligiera al ganador dentro de una sola rama, sin llegar a ver la otra. Y no
# cuesta ninguna codificación extra: truncar y evaluar es el eje gratis de D10.
BARRIDO = []
RANKINGS_DENSOS = {}
for (alias, contrato), vectores in VECTORES.items():
    for dim in REGISTRO[alias]["dims"]:
        informe, rankings = evaluar(vectores, dim)
        RANKINGS_DENSOS[(alias, contrato, dim)] = rankings
        BARRIDO.append({
            "modelo": alias,
            "contrato": contrato,
            # Etiqueta única de la configuración: `modelo` ya no la identifica,
            # porque ahora hay dos filas por modelo y dimensión.
            "sistema": f"{alias} [{contrato}]",
            "dim": dim,
            **informe.summary,
            "bytes_por_vector": dim * 4,
            "segundos": float(segundos_por_modelo.get((alias, contrato), float("nan"))),
        })

barrido = pd.DataFrame(BARRIDO).sort_values("ndcg_at_10", ascending=False)

# Qué ha entrado en el barrido, en voz alta: es la única forma de que quien lea
# el notebook sepa sobre qué se está decidiendo sin auditar el diccionario.
print(f"Evaluadas {len(barrido)} configuraciones sobre {len(query_ids)} consultas:")
for (alias, contrato), grupo in barrido.groupby(["modelo", "contrato"]):
    dims = ", ".join(str(d) for d in sorted(grupo["dim"], reverse=True))
    print(f"  · {alias:<16} [{contrato:<12}]  dims: {dims}")

faltan_ramas = [
    alias for alias, f in REGISTRO.items()
    if (f.get("api") or f.get("tasks")) and (alias, "sin_contrato") not in VECTORES
]
if faltan_ramas:
    print(
        f"\n⚠️  Sin la rama `sin_contrato`: {', '.join(faltan_ramas)}. Tienen contrato de"
        "\n    entrada, así que el barrido está incompleto y D09b decidiría sobre media"
        "\n    tabla. Ejecuta su celda de B.2 entera y vuelve aquí."
    )

barrido

Evaluadas 26 configuraciones sobre 8 consultas:
  · gemini-2         [nativo      ]  dims: 3072, 1536, 768, 512, 256, 128
  · gemini-2         [sin_contrato]  dims: 3072, 1536, 768, 512, 256, 128
  · granite-311m-r2  [nativo      ]  dims: 768, 512, 256, 128
  · jina-v3          [nativo      ]  dims: 1024, 768, 512, 256, 128
  · jina-v3          [sin_contrato]  dims: 1024, 768, 512, 256, 128


,modelo,contrato,sistema,dim,precision_at_10,recall_at_10,mrr_at_10,ndcg_at_10,bytes_por_vector,segundos
20,gemini-2,sin_contrato,gemini-2 [sin_contrato],3072,0.7875,0.4155,1.0000,0.7765,12288,49.59
21,gemini-2,sin_contrato,gemini-2 [sin_contrato],1536,0.7875,0.4155,1.0000,0.7750,6144,49.59
22,gemini-2,sin_contrato,gemini-2 [sin_contrato],768,0.7875,0.4155,1.0000,0.7718,3072,49.59
14,gemini-2,nativo,gemini-2 [nativo],3072,0.7625,0.3759,1.0000,0.7509,12288,45.41
15,gemini-2,nativo,gemini-2 [nativo],1536,0.7625,0.3759,1.0000,0.7496,6144,45.41
16,gemini-2,nativo,gemini-2 [nativo],768,0.7750,0.3795,1.0000,0.7478,3072,45.41
23,gemini-2,sin_contrato,gemini-2 [sin_contrato],512,0.7875,0.4155,1.0000,0.7410,2048,49.59
17,gemini-2,nativo,gemini-2 [nativo],512,0.7375,0.3688,0.9167,0.7110,2048,45.41
18,gemini-2,nativo,gemini-2 [nativo],256,0.7125,0.3297,1.0000,0.7101,1024,45.41
24,gemini-2,sin_contrato,gemini-2 [sin_contrato],256,0.7375,0.3965,0.9375,0.6843,1024,49.59


### C.1 · Curva calidad ↔ dimensión

Los mismos números del barrido, en la forma que responde a la pregunta real: **¿dónde se cae la curva?** Una tabla obliga a restar de cabeza para verlo; la línea lo enseña de un vistazo — si la caída es suave, MRL está funcionando; si hay un escalón, esa dimensión ya no basta para este catálogo.

Tres elementos del gráfico que no son decoración:

- **Color = modelo · trazo = contrato.** Son dos ejes cruzados. Metiéndolos los dos en el color saldrían cinco tonos sin relación aparente entre sí; separándolos, el ojo agrupa primero por modelo y luego compara la línea continua con la discontinua **dentro** de cada uno. Esa comparación —el mismo modelo consigo mismo— es la que importa, y es la que la sección D cuantifica.
- **La banda gris es la tolerancia τ = 0,02 de D09b.** Todo punto que cae dentro es *admisible* por la regla, y entre los admisibles gana el de menor dimensión: es decir, **el punto admisible situado más a la izquierda**. La sección G lo calcula formalmente con `apply_tolerance_rule`; aquí se ve venir antes de aplicarlo.
- **El eje X está en escala logarítmica** porque las dimensiones se barren dividiendo por dos. En escala lineal, 128 y 256 se amontonarían contra el margen izquierdo y la parte interesante de la curva —justo donde se decide el ahorro— quedaría ilegible.

Toda la lógica vive en `aurum.graficas`, cubierta por `tests/test_graficas.py`. El notebook solo declara *qué* quiere ver, no *cómo* se dibuja.

In [21]:
plot_dimension_curve(
    barrido,
    model_column="modelo",     # el color agrupa por modelo
    dash_column="contrato",    # el trazo separa con/sin contrato dentro de cada uno
    tolerance=TOLERANCIA_D09B,
    subtitle=(
        f"{CORPUS_ID} ({len(corpus_textos)} docs) · {len(query_ids)} consultas · "
        f"la banda gris es la tolerancia τ={TOLERANCIA_D09B} de D09b"
    ),
).show()

### C.2 · Tabla por consulta — la media esconde el caso 33633

NB00 midió que la consulta **33633** (*disfraz halloween talla grande hombre*) tiene **un solo `Exact`** en todo el pool: su Recall@10 solo puede valer 0 o 1, y una media macro sobre 8 consultas se mueve 0,125 según caiga. Reportar solo la media dejaría que esa consulta decidiera el modelo.

In [22]:
# Mejor configuración de cada modelo, ya **entre las dos ramas de contrato**:
# si `sin_contrato` gana, es esa la que representa al modelo de aquí en adelante.
mejor_por_modelo = barrido.loc[barrido.groupby("modelo")["ndcg_at_10"].idxmax()]

por_consulta = pd.concat([
    # `fila.contrato` y no "nativo" a mano: si se fijara, se leerían los vectores
    # de la otra rama sin ningún error visible —`(alias, "nativo")` también
    # existe— y la tabla mostraría por consulta un sistema que no es el que ganó.
    evaluar(VECTORES[(fila.modelo, fila.contrato)], int(fila.dim))[0]
    .per_query_frame()
    .assign(sistema=f"{fila.sistema}@{int(fila.dim)}")
    for fila in mejor_por_modelo.itertuples()
])
por_consulta.pivot(index="query_id", columns="sistema", values="ndcg@10")

sistema,gemini-2 [sin_contrato]@3072,granite-311m-r2 [nativo]@768,jina-v3 [sin_contrato]@512
query_id,,,
13357,0.7262,0.7384,0.6225
18868,0.4653,0.5171,0.7190
28703,0.9062,0.9266,0.7114
31224,0.7712,0.0590,0.6835
33633,0.6796,0.2414,0.0311
38249,0.8493,0.6236,0.1585
43240,0.9133,0.8018,0.8464
61533,0.9008,0.9052,0.6394


---

# D · El contrato de entrada (eje "prefijos" de D10)

> 📌 **Esta sección no codifica nada.** Las dos variantes de cada modelo se generaron en **B.2**, junto al resto de codificaciones. Aquí solo se comparan. Si has ejecutado B.2 completo, esta sección corre en segundos.

§3.1 del enunciado pide elegir cuatro cosas y justificarlas juntas:

> *"Después elegid **la representación textual, el modelo de embeddings, los prefijos que requiera y la normalización**. No basta con citar la documentación del modelo: la elección debe apoyarse en los resultados de desarrollo y en las restricciones del caso."*

El sujeto de "la elección" es esa enumeración entera, así que la exigencia se reparte por todo NB02: la **representación textual** está congelada en A0 y documentada en NB01/NB03, el **modelo** se decide con el barrido de C y la regla de G, la **normalización** se mide en E — y los **prefijos** son esta sección.

Lo que se hace aquí es lo que convierte una cita en evidencia: comparar cada modelo consigo mismo, con y sin su contrato.

- Si retirar el contrato **no cambia nada** (Δ ≈ 0), es que no se estaba aplicando: un fallo de integración disfrazado de resultado.
- Si **cambia**, queda demostrado con datos que el contrato hace algo — y el signo dice si ayuda o estorba en *este* catálogo, que no tiene por qué coincidir con lo que promete la model card.

Qué significa "sin contrato" en cada uno:

| Modelo | `nativo` | `sin_contrato` |
|---|---|---|
| `jina-v3` | Adaptador LoRA `retrieval.passage` / `retrieval.query` | Sin adaptador de tarea — **otros pesos** |
| `gemini-2` | Instrucción de tarea antepuesta al texto | Texto desnudo |
| `granite-311m-r2` | — | **No aplica**: sus dos prompts declarados son cadena vacía |

> ✅ **P02, cerrado.** Esa última exclusión era justo la que §3.1 no admite tal cual: se apoyaba en el `config_sentence_transformers.json` de granite, es decir, en **la documentación del modelo**. Se cerró leyendo la fuente primaria: la model card completa de IBM, con todos sus ejemplos de uso y las secciones *Usage* y *When to Use This Model*. **No documenta ninguna instrucción ni prefijo en ningún backend** — en su ejemplo de retrieval cross-lingual, `input_queries` e `input_passages` van directos a `model.encode()` sin nada antepuesto, a diferencia de `e5-instruct` o los BGE con instrucciones. El contrato de entrada real es **texto plano simétrico**, así que granite no compitió en desventaja y su exclusión de este eje queda sostenida por evidencia más fuerte que la que la motivó.

### D.2 · Δ nDCG@10 al retirar el contrato

In [23]:
filas = []
for alias in REGISTRO:
    if (alias, "sin_contrato") not in VECTORES:
        continue
    dim = REGISTRO[alias]["dim_nativa"]
    con, _ = evaluar(VECTORES[(alias, "nativo")], dim)
    sin, _ = evaluar(VECTORES[(alias, "sin_contrato")], dim)
    filas.append({
        "modelo": alias,
        "dim": dim,
        "ndcg_con_contrato": con.summary["ndcg_at_10"],
        "ndcg_sin_contrato": sin.summary["ndcg_at_10"],
        "delta": round(con.summary["ndcg_at_10"] - sin.summary["ndcg_at_10"], 4),
    })

pd.DataFrame(filas) if filas else "Sin modelos con contrato codificados"


,modelo,dim,ndcg_con_contrato,ndcg_sin_contrato,delta
0,jina-v3,1024,0.5055,0.5344,-0.0289
1,gemini-2,3072,0.7509,0.7765,-0.0256


---

# E · Normalización y métrica — qué significa el score

El enunciado (§3.2) pide *"conservar la semántica del score nativo"* y §3.1 *"explicar la relación entre la métrica configurada, la normalización y el significado del score"*. Estas dos celdas son esa explicación, medida:

- Con vectores **L2-normalizados**, `cosine` y `dot` producen el **mismo ranking** (el producto escalar de dos unitarios *es* el coseno), y `l2` también, porque `‖a−b‖² = 2 − 2·a·b` es una función monótona decreciente del producto escalar.
- **Sin normalizar**, `dot` premia los vectores de norma grande y el ranking cambia. Ese es el fallo silencioso que esta comprobación caza.

Si las tres filas normalizadas no coinciden, la normalización no se está aplicando y **todas las métricas del notebook están en duda**.

### ⚠️ Cómo leer la tabla: solo un modelo demuestra algo

Como se vio en B.3, `jina-v3` y `gemini-2` ya entregan vectores unitarios. Su fila *"sin normalizar"* recibe vectores unitarios igualmente, así que sale **idéntica** a la normalizada: no demuestra nada, solo confirma que normalizar dos veces es idempotente.

**Toda la evidencia de esta sección la aporta `granite-311m-r2`**, precisamente por ser el único que llega crudo. Ahí sí se ve el fenómeno: `cosine` no se mueve —normaliza internamente, es inmune— mientras `dot` baja y `l2` sube, y las tres métricas dejan de coincidir.

Y lo llamativo es **cuán poco hace falta para romperlo**: las normas de granite se desvían apenas unas milésimas de 1, y eso ya basta para reordenar resultados y mover el nDCG. No hace falta una anomalía grande para que `dot` deje de ser una medida de similitud.

> Si los tres modelos normalizaran en origen, esta comprobación pasaría sin detectar nada. Es exactamente el modo en que este tipo de verificación falla en silencio.

In [32]:
filas_semantica = []
for (alias, contrato), vectores in VECTORES.items():
    # Solo la rama `nativo`: lo que se demuestra aquí es una propiedad geométrica
    # de los vectores (con norma 1, coseno·dot·l2 ordenan igual), y esa propiedad
    # no depende del contrato con que se generaran. Recorrer las dos ramas
    # duplicaría las filas de la tabla sin añadir ninguna información nueva.
    if contrato != "nativo":
        continue
    dim = REGISTRO[alias]["dim_nativa"]
    for normalizar in (True, False):
        rankings_por_metrica, ndcg_por_metrica = {}, {}
        for metric in ("cosine", "dot", "l2"):
            informe, rankings = evaluar(vectores, dim, metric=metric, normalizar=normalizar)
            rankings_por_metrica[metric] = rankings
            ndcg_por_metrica[metric] = informe.summary["ndcg_at_10"]
        iguales = (
            rankings_por_metrica["cosine"] == rankings_por_metrica["dot"] == rankings_por_metrica["l2"]
        )
        # `mismo_ranking` se guarda en la fila, no solo se imprime: es la
        # conclusión de la sección y tiene que sobrevivir a un reinicio.
        filas_semantica += [
            {"modelo": alias, "dim": dim, "normalizado": normalizar,
             "metrica": metric, "ndcg_at_10": valor, "mismo_ranking": iguales}
            for metric, valor in ndcg_por_metrica.items()
        ]
        print(f"{alias} · normalizado={normalizar}: ¿mismo ranking en las 3 métricas? {iguales}")

semantica_score = pd.DataFrame(filas_semantica)
semantica_score.pivot(index=["modelo", "normalizado"], columns="metrica", values="ndcg_at_10")

jina-v3 · normalizado=True: ¿mismo ranking en las 3 métricas? True
jina-v3 · normalizado=False: ¿mismo ranking en las 3 métricas? True
granite-311m-r2 · normalizado=True: ¿mismo ranking en las 3 métricas? True
granite-311m-r2 · normalizado=False: ¿mismo ranking en las 3 métricas? False
gemini-2 · normalizado=True: ¿mismo ranking en las 3 métricas? True
gemini-2 · normalizado=False: ¿mismo ranking en las 3 métricas? True


metrica                      cosine     dot      l2
modelo          normalizado                        
gemini-2        False        0.7509  0.7509  0.7509
                True         0.7509  0.7509  0.7509
granite-311m-r2 False        0.6016  0.5965  0.6021
                True         0.6016  0.6016  0.6016
jina-v3         False        0.5055  0.5055  0.5055
                True         0.5055  0.5055  0.5055

---

# F · Contra el baseline léxico — el requisito del enunciado §3.1

> *"El trabajo debe comparar el sistema denso con, al menos, un baseline léxico o exacto."*

NB01 dejó ese baseline medido en `artifacts/baseline_lexico.json`. Aquí se recupera **sobre el mismo corpus** (la muestra de 1.500), con las mismas 8 consultas, el mismo `k`, los mismos qrels y el mismo contrato de relevancia. Sin esa igualdad no se compararían métodos, sino entornos (Regla 2).

La pregunta que responde la tabla no es *"¿gana el denso?"* sino **"¿cuánto gana y a cambio de qué coste?"**: BM25 se construye en segundos sobre CPU y no necesita ni modelo ni GPU ni base vectorial. Si la mejora del denso fuera marginal, el argumento de negocio para montar toda esta infraestructura sería flojo — y decirlo con un número es mejor informe que esconderlo.

In [24]:
import json

baseline = json.loads(
    (Path("..") / "artifacts" / "baseline_lexico.json").read_text(encoding="utf-8")
)
lexico_muestra = baseline["muestra"]["metricas"]

# `modelo` y `contrato` viajan hasta aquí aunque no se muestren: F.1 los necesita
# para volver a buscar los vectores del ganador en VECTORES. `sistema` es solo
# la etiqueta legible.
comparativa = pd.concat([
    pd.DataFrame([
        {"sistema": nombre, "familia": "léxico", "modelo": None, "contrato": None,
         "dim": None, **metricas}
        for nombre, metricas in lexico_muestra.items()
    ]),
    mejor_por_modelo.assign(familia="denso")[
        ["sistema", "familia", "modelo", "contrato", "dim",
         "precision_at_10", "recall_at_10", "mrr_at_10", "ndcg_at_10"]
    ],
]).sort_values("ndcg_at_10", ascending=False).reset_index(drop=True)

mejor_lexico = max(m["ndcg_at_10"] for m in lexico_muestra.values())
comparativa["delta_vs_mejor_lexico"] = (comparativa["ndcg_at_10"] - mejor_lexico).round(4)
comparativa

,sistema,familia,modelo,contrato,dim,precision_at_10,recall_at_10,mrr_at_10,ndcg_at_10,delta_vs_mejor_lexico
0,gemini-2 [sin_contrato],denso,gemini-2,sin_contrato,3072,0.7875,0.4155,1.0000,0.7765,0.1253
1,bm25,léxico,None,None,None,0.7125,0.3130,0.8906,0.6512,0.0000
2,granite-311m-r2 [nativo],denso,granite-311m-r2,nativo,768,0.6125,0.2554,0.9167,0.6016,-0.0496
3,tfidf,léxico,None,None,None,0.6000,0.2324,0.8750,0.5654,-0.0858
4,jina-v3 [sin_contrato],denso,jina-v3,sin_contrato,512,0.6000,0.2724,0.7750,0.5515,-0.0997


In [25]:
# La misma comparativa de arriba en barras agrupadas: las cuatro métricas en una
# escala 0-1 común, que es la forma en que §3.1 pide leer denso frente a léxico.
METRICAS = ["precision_at_10", "recall_at_10", "mrr_at_10", "ndcg_at_10"]


def etiqueta(fila):
    """El denso lleva su dimensión en el nombre: `granite-311m-r2@256` y `@768`
    son sistemas distintos y la leyenda tiene que poder distinguirlos."""
    if fila["familia"] == "léxico":
        return fila["sistema"]
    return f"{fila['sistema']}@{int(fila['dim'])}"


sistemas = {
    etiqueta(fila): {metrica: float(fila[metrica]) for metrica in METRICAS}
    for _, fila in comparativa.iterrows()
}

plot_metric_comparison(
    sistemas,
    title="Denso frente al baseline léxico de NB01",
    subtitle=(
        f"{CORPUS_ID} ({len(corpus_textos)} docs) · {len(query_ids)} consultas · k={TOP_K} "
        "· mismo corpus, mismos qrels, mismo contrato de relevancia"
    ),
).show()

### F.1 · Dónde gana cada familia, consulta a consulta

El agregado dice quién gana; esta tabla dice **por qué**. Lo interesante son las consultas donde el denso mejora mucho (vocabulario distinto al del catálogo) y las que empeora (el léxico acierta por coincidencia literal y el denso trae vecinos semánticamente próximos pero comercialmente distintos). Los dos casos alimentan la atribución de errores de NB09.

In [26]:
mejor_denso = comparativa.query("familia == 'denso'").iloc[0]
nombre_lexico = max(lexico_muestra, key=lambda n: lexico_muestra[n]["ndcg_at_10"])
etiqueta_densa = f"{mejor_denso['sistema']}@{int(mejor_denso['dim'])}"

ndcg_lexico = {
    str(fila["query_id"]): fila["ndcg@10"]
    for fila in baseline["muestra"]["por_consulta"][nombre_lexico]
}
# La clave de VECTORES es (modelo, contrato). `sistema` es la etiqueta legible
# —"jina-v3 [sin_contrato]"— y no sirve para buscar aquí.
informe_denso, _ = evaluar(
    VECTORES[(mejor_denso["modelo"], mejor_denso["contrato"])], int(mejor_denso["dim"])
)

frente_a_frente = (
    informe_denso.per_query_frame()
    .assign(
        query_id=lambda d: d["query_id"].astype(str),
        **{nombre_lexico: lambda d: d["query_id"].map(ndcg_lexico)},
    )
    .rename(columns={"ndcg@10": etiqueta_densa})
    [["query_id", nombre_lexico, etiqueta_densa]]
    .merge(consultas.assign(query_id=consultas["query_id"].astype(str)), on="query_id")
)
frente_a_frente["delta"] = (
    frente_a_frente[etiqueta_densa] - frente_a_frente[nombre_lexico]
).round(4)
frente_a_frente[["query_id", "query_text", nombre_lexico, etiqueta_densa, "delta"]].sort_values("delta")

,query_id,query_text,bm25,gemini-2 [sin_contrato]@3072,delta
7,61533,lentejas sin gluten,1.0000,0.9008,-0.0992
1,18868,botines marrones mujer tacon medio,0.5270,0.4653,-0.0617
2,28703,convertibles 2 en 1 portátil tactil,0.8709,0.9062,0.0353
5,38249,estantes sin taladro habitacion,0.8125,0.8493,0.0368
0,13357,base tapizada 160x200 sin patas,0.5441,0.7262,0.1821
6,43240,funda ipad air 4 sin tapa,0.7208,0.9133,0.1925
3,31224,cámaras bridge baratas,0.5777,0.7712,0.1935
4,33633,disfraz halloween talla grande hombre,0.1566,0.6796,0.5230


---

# G · R02 · Aplicar D09b y dejar el artefacto

**D09b se fijó en `config/config.yaml` antes de codificar nada.** Aplicarla como función y no a ojo es lo que hace verificable esa afirmación: el ganador sale de `apply_tolerance_rule`, que es determinista y está cubierta por tests.

```yaml
d09b_criterio_desempate:
  forma: mas_barata_dentro_de_tolerancia
  metrica_primaria: ndcg_at_10
  tolerancia_tau: 0.02
```

1. `B` = mejor nDCG@10 de toda la tabla.
2. Admisibles: las que están a menos de 0,02 de `B` — con 8 consultas, una diferencia menor no distingue dos sistemas.
3. Entre las admisibles gana **la de menor dimensión**; a igualdad, mayor nDCG; después, menor tiempo de codificación.

> 🗳️ **Paso 6 del bucle — te toca a ti.** La celda produce la ordenación; **R02 la ratificas tú** y la escribes en `config/config.yaml`. Si el resultado te parece equivocado, el sitio para discutirlo es el criterio, no la tabla: cambiar la regla después de ver los números es exactamente lo que el enunciado penaliza.

In [27]:
ordenadas = apply_tolerance_rule(
    barrido, metrica="ndcg_at_10", tolerancia=TOLERANCIA_D09B,
    coste="dim", desempates=("segundos",),
)
# `contrato` en las columnas mostradas: sin él, dos configuraciones distintas
# del mismo modelo y dimensión aparecen como filas idénticas y no hay forma de
# saber cuál ha ganado.
ordenadas[["posicion_regla", "modelo", "contrato", "dim", "ndcg_at_10", "recall_at_10",
           "mrr_at_10", "bytes_por_vector", "segundos", "admisible"]]

,posicion_regla,modelo,contrato,dim,ndcg_at_10,recall_at_10,mrr_at_10,bytes_por_vector,segundos,admisible
0,1,gemini-2,sin_contrato,768,0.7718,0.4155,1.0000,3072,49.59,True
1,2,gemini-2,sin_contrato,1536,0.7750,0.4155,1.0000,6144,49.59,True
2,3,gemini-2,sin_contrato,3072,0.7765,0.4155,1.0000,12288,49.59,True
3,4,gemini-2,nativo,128,0.6648,0.3036,1.0000,512,45.41,False
4,5,gemini-2,sin_contrato,128,0.5575,0.2888,0.9000,512,49.59,False
5,6,jina-v3,sin_contrato,128,0.5330,0.2724,0.7656,512,5751.93,False
6,7,granite-311m-r2,nativo,128,0.5164,0.2588,0.7500,512,17489.67,False
7,8,jina-v3,nativo,128,0.4961,0.2756,0.6429,512,4954.58,False
8,9,gemini-2,nativo,256,0.7101,0.3297,1.0000,1024,45.41,False
9,10,gemini-2,sin_contrato,256,0.6843,0.3965,0.9375,1024,49.59,False


In [28]:
def registros(frame):
    """Filas como tipos JSON nativos: `to_dict` dejaría escalares de numpy."""
    return json.loads(frame.to_json(orient="records"))


artefacto = {
    "configuracion": {
        "corpus": CORPUS_ID,
        "n_docs": len(corpus_textos),
        "plantilla": PLANTILLA,
        "campo": CAMPO,
        "top_k": TOP_K,
        "relevancia": {"E": 3, "S": 2, "C": 1, "I": 0},
        "d09b": {"metrica": "ndcg_at_10", "tolerancia": TOLERANCIA_D09B, "coste": "dim"},
    },
    "modelos": {alias: {k: v for k, v in f.items() if k != "tasks"} for alias, f in REGISTRO.items()},
    "errores_de_codificacion": ERRORES,
    "costes_de_codificacion": list(COSTES.values()),
    "modelos_codificados": [f"{a}[{c}]" for a, c in sorted(VECTORES)],
    "barrido": registros(barrido),
    "regla_d09b": registros(ordenadas),
    "comparativa_con_lexico": registros(comparativa),
    # §3.1 pide explicar la relación entre métrica, normalización y score. Sin
    # esto, esa evidencia vivía solo en la salida de una celda.
    "semantica_del_score": registros(semantica_score),
    # La clave lleva el contrato: el barrido tiene ahora dos rankings por modelo
    # y dimensión, y sin él uno sobrescribiría al otro en silencio.
    "rankings": {
        f"{alias}[{contrato}]@{dim}": r
        for (alias, contrato, dim), r in RANKINGS_DENSOS.items()
    },
}

destino = Path("..") / "artifacts" / "comparativa_modelos.json"
destino.write_text(json.dumps(artefacto, indent=2, ensure_ascii=False, default=str), encoding="utf-8")

markdown = Path("..") / "artifacts" / "comparativa_modelos.md"
markdown.write_text(
    "# Comparativa de modelos (NB02)\n\n"
    f"Corpus: `{CORPUS_ID}` ({len(corpus_textos)} docs) · plantilla `{PLANTILLA}` · k={TOP_K}\n\n"
    "## Barrido modelo x contrato x dimension\n\n" + barrido.to_markdown(index=False) + "\n\n"
    "## Regla D09b aplicada\n\n" + ordenadas.to_markdown(index=False) + "\n\n"
    "## Denso frente al baseline lexico de NB01\n\n" + comparativa.to_markdown(index=False) + "\n",
    encoding="utf-8",
)
print(f"Escrito {destino.name} ({destino.stat().st_size / 1024:.1f} KB) y {markdown.name}")

Escrito comparativa_modelos.json (76.6 KB) y comparativa_modelos.md


---

# H · El contrato no aporta lo mismo en todas las dimensiones

La sección D midió la Δ del contrato **solo en la dimensión nativa**, y con ese único punto la conclusión parecía limpia: retirarlo mejora en los dos modelos que lo tienen. El barrido completo dice algo más incómodo — y más interesante.

**La Δ cambia de signo al truncar.** En `gemini-2`:

| dim | `nativo` | `sin_contrato` | Δ |
|---:|---:|---:|---:|
| 3072 | 0,7509 | 0,7765 | **−0,0256** |
| 1536 | 0,7496 | 0,7750 | −0,0254 |
| 768 | 0,7478 | 0,7718 | −0,0240 |
| 512 | 0,7110 | 0,7410 | −0,0300 |
| 256 | 0,7101 | 0,6843 | **+0,0258** |
| 128 | 0,6648 | 0,5575 | **+0,1073** |

Por encima de 512 el contrato estorba. Por debajo, ayuda — y a 128 la diferencia es de **0,107**, cinco veces la tolerancia de D09b. El cruce está entre 512 y 256.

### Una hipótesis, no una conclusión

La instrucción de tarea es **texto idéntico en los 1.500 documentos**. En un modelo entrenado con MRL, las primeras componentes concentran la estructura más gruesa y compartida del corpus, y ahí es donde esa señal común pesa más.

- **A dimensión completa**, ese prefijo compartido es sobre todo lastre: ocupa norma sin aportar nada que distinga un producto de otro, así que quitarlo mejora.
- **Al truncar fuerte**, te quedas casi solo con esas primeras componentes, y ahí la instrucción funciona como condicionamiento de tarea que sí orienta la búsqueda.

No está comprobado — habría que mirar la energía por componente en ambas variantes. Queda como hipótesis explícita, no como explicación cerrada.

### Consecuencia práctica

**"Sin contrato es mejor" no es incondicional: vale a partir de 512.** La configuración que gana D09b (`gemini-2 [sin_contrato] @768`) cae con holgura en esa zona, así que **R02 no se ve afectada**.

Pero es una condición que hay que arrastrar. Si más adelante la memoria del índice empujara a bajar de dimensión —15.000 productos a 768 son 46 MB; a 256 serían 15 MB— la decisión sobre el contrato tendría que **revisarse, no heredarse**. Medir ese eje en un solo punto habría dejado la trampa puesta.

In [29]:
from aurum.graficas import plot_contract_delta

plot_contract_delta(
    barrido,
    tolerance=TOLERANCIA_D09B,
    subtitle=(
        f"{CORPUS_ID} ({len(corpus_textos)} docs) · {len(query_ids)} consultas · "
        f"dentro de la banda gris (±{TOLERANCIA_D09B}) la diferencia no se distingue"
    ),
).show()

# El cruce, en números: dónde deja de convenir retirar el contrato.
cruce = (
    barrido.pivot_table(index=["modelo", "dim"], columns="contrato", values="ndcg_at_10")
    .dropna(subset=["nativo", "sin_contrato"])
    .assign(delta=lambda d: (d["nativo"] - d["sin_contrato"]).round(4))
    .assign(gana=lambda d: d["delta"].apply(
        lambda x: "nativo" if x > TOLERANCIA_D09B
        else ("sin_contrato" if x < -TOLERANCIA_D09B else "indistinguible")
    ))
    .reset_index()
    .sort_values(["modelo", "dim"], ascending=[True, False])
)
cruce[["modelo", "dim", "nativo", "sin_contrato", "delta", "gana"]]

contrato,modelo,dim,nativo,sin_contrato,delta,gana
5,gemini-2,3072,0.7509,0.7765,-0.0256,sin_contrato
4,gemini-2,1536,0.7496,0.7750,-0.0254,sin_contrato
3,gemini-2,768,0.7478,0.7718,-0.0240,sin_contrato
2,gemini-2,512,0.7110,0.7410,-0.0300,sin_contrato
1,gemini-2,256,0.7101,0.6843,0.0258,nativo
0,gemini-2,128,0.6648,0.5575,0.1073,nativo
10,jina-v3,1024,0.5055,0.5344,-0.0289,sin_contrato
9,jina-v3,768,0.5083,0.5398,-0.0315,sin_contrato
8,jina-v3,512,0.5188,0.5515,-0.0327,sin_contrato
7,jina-v3,256,0.5105,0.5329,-0.0224,sin_contrato


---

# I · P01 · El ganador sobre el catálogo completo

Todo lo anterior está medido sobre **1.500 documentos**, la muestra de la condición 3 del plan. El enunciado (§6, *Condiciones de comparabilidad*) dice que *"el catálogo completo es el recorrido evaluado; la muestra sirve para desarrollar y depurar"*. Esta sección cierra esa distancia para la configuración que ganó D09b.

### Por qué esto no es un detalle

NB01 ya midió lo que pasa al multiplicar por diez los candidatos, con **los mismos juicios de relevancia**:

| baseline | muestra (1.500) | completo (15.000) | caída |
|---|---:|---:|---:|
| BM25 | 0,6512 | 0,5088 | **−0,1424** |
| TF-IDF | 0,5654 | 0,4129 | −0,1525 |

Los qrels no cambian: lo que cambia es que aparecen 13.500 productos más que compiten por las 10 posiciones y que, al no estar juzgados, puntúan 0 (D04). Un sistema que sube documentos no juzgados se desploma; uno que mantiene arriba los juzgados aguanta. **Es un test de precisión bajo distracción**, y no hay forma de aprobarlo desde la muestra.

### Qué se ejecuta aquí, y qué no

Solo se codifica el **ganador de D09b**, y la celda lo lee de `ordenadas` en vez de escribirlo a mano: si la regla cambiara de ganador, esta sección lo sigue.

El motivo es de coste, y está medido, no estimado a ojo:

| configuración | 1.500 docs | 15.000 (×10) |
|---|---:|---:|
| `gemini-2` | ~50 s | **~8 min** ✅ |
| `jina-v3` | ~5.750 s | ~16 h ❌ |
| `granite-311m-r2` | ~17.490 s | ~49 h ❌ |

La celda calcula esa extrapolación y **se niega a lanzar** cualquier codificación por encima de `LIMITE_HORAS`. Si el ganador fuera un modelo local, imprimiría el coste y se saltaría el paso en lugar de dejar el kernel bloqueado media semana.

Dos cosas más que abaratan la prueba:

- **Las consultas no se recodifican.** Su caché es independiente del corpus (`corpus_id="consultas_desarrollo"`), así que los 8 vectores ya están en disco.
- **Las tres dimensiones admisibles salen de la misma codificación.** Truncar es gratis, así que 768, 1.536 y 3.072 se evalúan sin ninguna llamada extra.

### Qué responde y qué no

Responde a la pregunta del enunciado: **¿el denso sigue batiendo al léxico cuando el catálogo es el de verdad?**

No responde al orden **entre modelos densos** a escala completa: para eso habría que pagar las 65 horas de jina y granite. Queda anotado como límite explícito del experimento — con `gemini-2` sacando 0,12 sobre BM25 y 0,24 sobre jina en la muestra, el riesgo de que el orden se invierta es bajo, pero *bajo* no es *cero* y el informe debe decirlo así.

In [30]:
LIMITE_HORAS = 1.0            # por encima de esto la celda no lanza la codificación
CORPUS_COMPLETO = "catalogo_productos"

# El ganador se lee de la regla, no se escribe a mano: si D09b cambiara de
# resultado, esta sección lo sigue sin tocar una línea.
ganador = ordenadas.iloc[0]
ALIAS_G = ganador["modelo"]
CONTRATO_G = ganador["contrato"]
DIM_G = int(ganador["dim"])
ETIQUETA_G = f"{ALIAS_G} [{CONTRATO_G}]@{DIM_G}"

textos_completo = completo[CAMPO].tolist()
ids_completo = completo["product_id"].tolist()

# Extrapolación lineal desde el coste ya medido sobre la muestra. Es fiable
# porque codificar es proporcional al número de documentos: mismo modelo, mismo
# lote, mismo hardware.
segundos_muestra = float(
    costes.query(
        "alias == @ALIAS_G and contrato == @CONTRATO_G and tipo == 'document'"
    )["segundos"].iloc[0]
)
horas = segundos_muestra * len(textos_completo) / len(corpus_textos) / 3600

print(f"Ganador de D09b : {ETIQUETA_G}")
print(f"Corpus completo : {len(textos_completo)} documentos")
print(f"Coste estimado  : ~{horas:.2f} h  ({segundos_muestra:.0f}s para {len(corpus_textos)} docs)")

if horas > LIMITE_HORAS:
    vectores_completo = None
    print(
        f"\n⏭️  Por encima del límite de {LIMITE_HORAS} h: no se lanza.\n"
        "    P01 queda abierto para esta configuración. Sube LIMITE_HORAS si\n"
        "    quieres pagarlo, pero hazlo sabiendo cuántas horas son."
    )
else:
    encoder = fabricar(ALIAS_G)
    try:
        resultado = encode_corpus(
            encoder, textos_completo, corpus_id=CORPUS_COMPLETO,
            kind="document", contract=CONTRATO_G,
            batch_size=32, cache_dir=CACHE,
        )
    finally:
        del encoder
        gc.collect()
    vectores_completo = resultado.vectors
    origen = "desde caché" if resultado.stats.desde_cache else f"{resultado.stats.segundos:.0f}s"
    print(f"\n✅ {vectores_completo.shape[0]} vectores de {vectores_completo.shape[1]} dims ({origen})")

Ganador de D09b : gemini-2 [sin_contrato]@768
Corpus completo : 15000 documentos
Coste estimado  : ~0.14 h  (50s para 1500 docs)

✅ 15000 vectores de 3072 dims (495s)


In [31]:
if vectores_completo is None:
    print("P01 sin cerrar: la celda anterior no codificó el catálogo completo.")
else:
    # Las consultas NO se recodifican: su caché es independiente del corpus.
    vectores_query = VECTORES[(ALIAS_G, CONTRATO_G)]["query"]

    def evaluar_completo(dim):
        """Igual que `evaluar`, pero contra los 15.000 IDs del catálogo entero.

        No se reutiliza `evaluar` porque aquella cerró sobre `corpus_ids`, que
        son los 1.500 de la muestra: pasarle estos vectores devolvería IDs
        equivocados sin dar ningún error."""
        docs = truncate_dim(vectores_completo, dim)
        queries = truncate_dim(vectores_query, dim)
        retriever = DenseRetriever(docs, ids_completo, metric="cosine")
        rankings = rank_queries_dense(retriever, query_ids, queries, k=TOP_K)
        return evaluate_rankings(rankings, qrels, k=TOP_K), rankings

    lexico_completo = baseline["completo"]["metricas"]
    ndcg_muestra = {n: m["ndcg_at_10"] for n, m in lexico_muestra.items()}

    filas = []
    RANKINGS_COMPLETO = {}
    # Las tres dimensiones admisibles de D09b salen de la misma codificación:
    # truncar es gratis, así que verlas todas no cuesta ninguna llamada extra.
    for dim in sorted(ordenadas.query("admisible")["dim"].unique(), reverse=True):
        informe, rankings = evaluar_completo(int(dim))
        RANKINGS_COMPLETO[int(dim)] = rankings
        etiqueta = f"{ALIAS_G} [{CONTRATO_G}]@{int(dim)}"
        ndcg_muestra[etiqueta] = float(
            barrido.query(
                "modelo == @ALIAS_G and contrato == @CONTRATO_G and dim == @dim"
            )["ndcg_at_10"].iloc[0]
        )
        filas.append({"sistema": etiqueta, "familia": "denso", **informe.summary})

    filas += [
        {"sistema": nombre, "familia": "léxico", **metricas}
        for nombre, metricas in lexico_completo.items()
    ]

    completo_vs_lexico = (
        pd.DataFrame(filas).sort_values("ndcg_at_10", ascending=False).reset_index(drop=True)
    )
    completo_vs_lexico["muestra_1500"] = completo_vs_lexico["sistema"].map(ndcg_muestra)
    completo_vs_lexico["caida_al_escalar"] = (
        completo_vs_lexico["ndcg_at_10"] - completo_vs_lexico["muestra_1500"]
    ).round(4)

    mejor_lexico_completo = max(m["ndcg_at_10"] for m in lexico_completo.values())
    ventaja = (
        completo_vs_lexico.query("familia == 'denso'").iloc[0]["ndcg_at_10"]
        - mejor_lexico_completo
    )
    print(
        f"Ventaja del mejor denso sobre el mejor léxico:\n"
        f"  muestra  (1.500) : {barrido.iloc[0]['ndcg_at_10'] - max(ndcg_muestra[n] for n in lexico_muestra):+.4f}\n"
        f"  completo (15.000): {ventaja:+.4f}"
    )
    display(
        completo_vs_lexico[
            ["sistema", "familia", "muestra_1500", "ndcg_at_10",
             "caida_al_escalar", "recall_at_10", "mrr_at_10", "precision_at_10"]
        ]
    )

    plot_metric_comparison(
        {
            fila["sistema"]: {m: float(fila[m]) for m in METRICAS}
            for _, fila in completo_vs_lexico.iterrows()
        },
        title="Denso frente al léxico — catálogo completo",
        subtitle=(
            f"{CORPUS_COMPLETO} ({len(textos_completo)} docs) · {len(query_ids)} consultas "
            f"· k={TOP_K} · mismos qrels que sobre la muestra"
        ),
    ).show()

Ventaja del mejor denso sobre el mejor léxico:
  muestra  (1.500) : +0.1253
  completo (15.000): +0.0806


,sistema,familia,muestra_1500,ndcg_at_10,caida_al_escalar,recall_at_10,mrr_at_10,precision_at_10
0,gemini-2 [sin_contrato]@768,denso,0.7718,0.5894,-0.1824,0.2632,0.8438,0.6250
1,gemini-2 [sin_contrato]@3072,denso,0.7765,0.5849,-0.1916,0.2561,0.8438,0.6000
2,gemini-2 [sin_contrato]@1536,denso,0.7750,0.5849,-0.1901,0.2561,0.8438,0.6000
3,bm25,léxico,0.6512,0.5088,-0.1424,0.1841,0.7500,0.5500
4,tfidf,léxico,0.5654,0.4129,-0.1525,0.1510,0.7500,0.4125
